# Pattern 1: ReAct Tool-Using Agent (LangChain Agents + Claude)

This notebook demonstrates a classic ReAct-style agent that reasons and uses tools.
It runs with Claude Haiku 4.5 via `ANTHROPIC_API_KEY` and LangChain.

```mermaid
flowchart LR
    U[User Prompt] --> A[Agent Reasoning]
    A -->|needs external action| T[Tool Call]
    T --> A
    A --> R[Final Answer]
```

In [1]:
from datetime import datetime
from typing import Any

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_anthropic import ChatAnthropic

MODEL_NAME = "claude-haiku-4-5-20251001"
llm = ChatAnthropic(model=MODEL_NAME, temperature=0, max_tokens=1024)

In [3]:
@tool
def current_time(_: str = "") -> str:
    """Return the current local timestamp."""
    return datetime.now().isoformat(timespec="seconds")

@tool
def safe_calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '(12 * 9) + 7'."""
    allowed = set("0123456789+-*/(). %")
    if any(ch not in allowed for ch in expression):
        return "Only arithmetic characters are allowed."
    try:
        value = eval(expression, {"__builtins__": {}}, {})
        return str(value)
    except Exception as exc:
        return f"Calculation error: {exc}"

tools = [current_time, safe_calculator]

In [4]:
agent = create_agent(model=llm, tools=tools)

prompt = (
    "What is (27 * 14) + 9? Then tell me the current local timestamp. "
    "Show both answers clearly."
)

result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
result["messages"][-1].content

'Here are your answers:\n\n**Arithmetic Calculation:**\n(27 * 14) + 9 = **387**\n\n**Current Local Timestamp:**\n**2026-04-15T14:13:44**'

## How the code cells map to the pattern
Cell 2 configures the Claude client and keeps the model deterministic with `temperature=0` so tool selection is reproducible.
Cell 3 defines the two tools the agent can choose from: one for time and one for safe arithmetic. This is the ReAct action space.
Cell 4 creates the agent, sends a prompt that requires both tools, and shows the final integrated answer after the model reasons over tool outputs.